# M14.6 — Sondas de información del banco de filtros sobre el residuo

Pre-registro: `docs/experiments/M14_6_bank_probes.md` (19-11-2026, antes de calcular).

- **Residuos:** r = y − μ en val G3T, unidades estandarizadas; primario M14.5b, secundario M14.5a.
- **Sondas:** A (μ, σ, 3 correlaciones de Σ del modelo) y B = A + variables del banco (P32, P128, P512, L32). HistGradientBoosting fijo, 5 particiones (`KFold(5, shuffle, random_state=0)`, idénticas para todo), R²_oof frente a 0.
- **Celdas:** global y D1–D3 / D4–D7 / D8–D10 de τ30 predicha por M14.5b (cortes en cal de M14.5b, como M14.5), mismas filas para 5a y 5b.
- **IC:** bootstrap de filas (2000) dentro de cada celda, emparejado (mismos remuestreos para todos los conjuntos, etiquetas y residuos).
- **Gates:** G3 (en el npz de variables), G1, G2. Si falla alguno, el notebook se detiene antes de las sondas principales.
- Test no se usa. Ejecutar en orden, en local, CPU, 1 hilo.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # before numpy / sklearn
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from datetime import datetime, timezone
import sys, json, hashlib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from IPython.display import display

root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src/paths.py").is_file()), None)
if root is None:
    raise RuntimeError("Abrir desde cbc_pe")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.paths import resolve_project_root, resolve_data_root, resolve_processed_artifact
from scripts.m13.m13_1_diagnostics import tau_newtonian   # same tau30 as M14.1-M14.5

PROJECT_ROOT = resolve_project_root()

# ---- Inputs (edit only FEATURES_FILE) ----
FEATURES_FILE = "M13_3/M14_6/features_val_<UTC>.npz"           # relative to <data_root>/results/
MODELS = {
    "5b": dict(tag="M14_5b_4a_G3T_tbank_fullcov_s123",
               train_cfg="configs/experiments/m14/train_m14_5b_4aG3T_tbank_fullcov_s123.json"),
    "5a": dict(tag="M14_5a_4a_G3T_fullcov_s123",
               train_cfg="configs/experiments/m14/train_m14_5a_4aG3T_fullcov_s123.json"),
}
PRIMARY = "5b"
M14_5B_SUMMARY_GLOB = "M13_3/M14_5/M14_5b_4a_G3T_tbank_fullcov_s123/M14_5_summary_*_20261113T150740Z.json"

# ---- Pre-registered settings ----
LABELS = ["chirp_mass", "total_mass", "chi_eff"]
HGB_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=100,
                  l2_regularization=1.0, early_stopping=True, validation_fraction=0.1, random_state=0)
N_FOLDS, FOLD_SEED = 5, 0
N_BOOT, CI_LEVEL = 2000, 0.95
N_TAU_DECILES = 10
TAU_GROUPS = {"D1-D3": [0, 1, 2], "D4-D7": [3, 4, 5, 6], "D8-D10": [7, 8, 9]}
BANKS = {"P32": ("P", 32), "P128": ("P", 128), "P512": ("P", 512), "L32": ("L", 32)}
G1_MIN, G2_MAX = 0.8, 0.005
CRIT = dict(model="5b", bank="P512", labels=["chirp_mass", "chi_eff"], cell="D8-D10", relevant_lo=0.04, none_hi=0.02)
# Implementation seeds fixed before computing (docs, "Decisiones de implementación"):
BOOT_SEED, PERM_SEED = 146001, 146002

def read_json(path):
    return json.loads(Path(path).read_text())
def require(condition, message):
    if not condition:
        raise ValueError(message)
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()
CHECKS = []
def check(ok, name):
    require(bool(ok), "Check failed: " + name)
    CHECKS.append(name)

cfg5b = read_json(PROJECT_ROOT / MODELS["5b"]["train_cfg"])
DATA_ROOT = resolve_data_root(config_data_root=cfg5b["data_root"])
RESULTS = DATA_ROOT / "results"
OUT_DIR = RESULTS / "M13_3/M14_6"
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LABEL_STATS_PATH = resolve_processed_artifact(data_root=DATA_ROOT, dataset_id=cfg5b["dataset"]["dataset_id"],
                                              file_name=cfg5b["dataset"]["label_stats_file"])
print("sklearn", sklearn.__version__, "| numpy", np.__version__, "| data root", DATA_ROOT, "| stamp", STAMP)

## 1. Predicciones de val (5b, 5a) y cal (5b, solo cortes de τ30); residuos y variables A

In [ ]:
with np.load(LABEL_STATS_PATH) as st:
    y_mean, y_std = st["y_mean"].astype(np.float32), st["y_std"].astype(np.float32)

def load_pred(tag, s):
    path = RESULTS / f"M13_3/{tag}/preds_{s}_G3T/{s}_predictions_embeddings.npz"
    with np.load(path, allow_pickle=True) as z:
        check(z["available_splits"].tolist() == [s], f"{tag}/{s}: only split {s}")
        check(z["label_names"].tolist() == LABELS, f"{tag}/{s}: label order")
        check(np.array_equal(z["y_mean"], y_mean) and np.array_equal(z["y_std"], y_std), f"{tag}/{s}: train-only scaler")
        check(Path(str(z["checkpoint_file"])).name == f"{tag}_final.pt", f"{tag}/{s}: checkpoint {tag}_final.pt")
        mk = z["model_config"].item()["model_kwargs"]
        check(mk.get("heteroscedastic") and mk.get("covariance") == "full", f"{tag}/{s}: full covariance")
        check(not z["input_normalization"].item().get("enabled", False), f"{tag}/{s}: X already normalized")
        d = dict(idx=np.asarray(z[f"idx_{s}"], np.int64), pred=np.asarray(z[f"pred_{s}"], float),
                 y=np.asarray(z[f"y_{s}"], float), sigma=np.asarray(z[f"sigma_{s}"], float),
                 cov=np.asarray(z[f"cov_{s}"], float), memmap_dir=str(z["memmap_dir"]))
    d["path"], d["sha256"] = str(path), file_sha256(path)
    return d

VAL = {m: load_pred(v["tag"], "val") for m, v in MODELS.items()}
CAL5B = load_pred(MODELS["5b"]["tag"], "cal")
n = len(VAL["5b"]["idx"])
check(n == 40000, "40 000 val rows")
check(np.array_equal(VAL["5a"]["idx"], VAL["5b"]["idx"]), "5a and 5b: same val rows, same order")
check(np.array_equal(VAL["5a"]["y"], VAL["5b"]["y"]), "5a and 5b: same y")
row_ids = VAL["5b"]["idx"]
check(np.array_equal(np.load(Path(VAL["5b"]["memmap_dir"]) / "row_ids_val.npy"), row_ids), "idx_val = memmap row_ids_val")

def corr3(cov):
    sd = np.sqrt(np.diagonal(cov, axis1=1, axis2=2))
    return np.stack([cov[:, i, j] / (sd[:, i] * sd[:, j]) for i, j in ((0, 1), (0, 2), (1, 2))], 1)

RESID = {m: d["y"] - d["pred"] for m, d in VAL.items()}            # standardized, (n, 3)
FEAT_A = {m: np.hstack([d["pred"], d["sigma"], corr3(d["cov"])]) for m, d in VAL.items()}   # (n, 9)
for m in MODELS:
    check(np.isfinite(FEAT_A[m]).all() and np.isfinite(RESID[m]).all(), f"{m}: finite A variables and residuals")

# tau30 groups: cut on cal predictions of 5b (as TAU_EDGES_CAL of M14.5), applied to val predictions of 5b.
tau_cal = tau_newtonian(CAL5B["pred"][:, 0] * y_std[0] + y_mean[0])
tau_val = tau_newtonian(VAL["5b"]["pred"][:, 0] * y_std[0] + y_mean[0])
TAU_EDGES_CAL = np.quantile(tau_cal, np.linspace(0, 1, N_TAU_DECILES + 1))
DEC = np.clip(np.searchsorted(TAU_EDGES_CAL[1:-1], tau_val, side="right"), 0, N_TAU_DECILES - 1)
summ = sorted(RESULTS.glob(M14_5B_SUMMARY_GLOB))
check(len(summ) == 1, "one M14.5b summary JSON")
check(np.allclose(TAU_EDGES_CAL, read_json(summ[0])["groups"]["tau_pred_edges_cal_s"], rtol=1e-12, atol=0),
      "tau30 edges = M14.5b summary (tau_pred_edges_cal_s)")
CELLS = {"global": np.ones(n, bool), **{g: np.isin(DEC, ds) for g, ds in TAU_GROUPS.items()}}
print("tau30 edges (cal 5b) [s]:", np.round(TAU_EDGES_CAL, 4))
print({g: int(m.sum()) for g, m in CELLS.items()})

## 2. Variables del banco (P512 ⊃ P128 ⊃ P32, L32) y G3

In [ ]:
FEATURES_PATH = RESULTS / FEATURES_FILE
with np.load(FEATURES_PATH, allow_pickle=False) as z:
    fmeta = json.loads(str(z["metadata_json"]))
    check(np.array_equal(z["row_ids"], row_ids), "features row_ids = idx_val")
    FP = np.asarray(z["features_P"], np.float32)       # (n, 512, 4): [H1, L1, V1, network]
    FL = np.asarray(z["features_L32"], np.float32)     # (n, 32, 4)
tmeta = fmeta["templates_metadata"]
check(FP.shape == (n, 512, 4) and FL.shape == (n, 32, 4), "feature shapes")
check(tmeta["n_templates"] == 512 and tmeta["seed"] == 20261119, "P512, seed 20261119")
check(Path(fmeta["checkpoint_file"]).name == f"{MODELS['5b']['tag']}_final.pt", "L32 from the M14.5b checkpoint")
check(fmeta["gate_G3"]["passed"] and fmeta["gate_G3"]["rel_diff"] <= 1e-4 and fmeta["gate_G3"]["rows"] == 64,
      "G3: reproduced bank layer = checkpoint forward (rel <= 1e-4, 64 rows)")
check(np.isfinite(FP).all() and np.isfinite(FL).all(), "finite bank features")
print("G3:", fmeta["gate_G3"])

def bank_features(name):
    kind, k = BANKS[name]
    F = FP[:, :k] if kind == "P" else FL[:, :k]      # nested P banks: first k templates of the draw
    return F.reshape(n, -1)

display(pd.DataFrame({b: [bank_features(b).shape[1]] for b in BANKS}, index=["n variables"]))
display(pd.DataFrame(np.percentile(FP[..., 3], [5, 50, 95], axis=0).mean(1)[None], columns=["P5", "P50", "P95"],
                     index=["pico de red P512 (media sobre plantillas)"]))

## 3. Sondas: particiones, R²_oof y bootstrap emparejado

In [ ]:
FOLDS = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=FOLD_SEED).split(np.arange(n)))
OOF, FIT_TIME = {}, {}

def oof_predict(F, target):
    out = np.empty(n)
    for tr, te in FOLDS:
        out[te] = HistGradientBoostingRegressor(**HGB_PARAMS).fit(F[tr], target[tr]).predict(F[te])
    return out

def get_oof(key, F_fn, target):
    """key = (residual model or 'y', label, variable set); computed once, deterministic."""
    if key not in OOF:
        t0 = time.perf_counter()
        OOF[key] = oof_predict(F_fn(), target)
        FIT_TIME[key] = time.perf_counter() - t0
        print(f"{key}: {FIT_TIME[key] / 60:.1f} min", flush=True)
    return OOF[key]

def r2(target, pred, mask=None):
    m = slice(None) if mask is None else mask
    return 1.0 - np.sum((target[m] - pred[m]) ** 2) / np.sum(target[m] ** 2)

def boot_r2(series, cells=CELLS):
    """series: {name: (target, oof)}. Paired bootstrap of rows within each cell (same draws for every series).
    Returns {cell: {name: (B,) R2 replicates}}; draws from default_rng([BOOT_SEED, cell index])."""
    out = {}
    names = list(series)
    for ci, (cell, mask) in enumerate(cells.items()):
        E = np.stack([(series[s][0][mask] - series[s][1][mask]) ** 2 for s in names])   # (S, n_g)
        R = np.stack([series[s][0][mask] ** 2 for s in names])
        rng = np.random.default_rng([BOOT_SEED, ci])
        n_g = int(mask.sum())
        res = np.empty((len(names), N_BOOT))
        for b0 in range(0, N_BOOT, 250):
            nb = min(250, N_BOOT - b0)
            W = np.stack([np.bincount(rng.integers(0, n_g, n_g), minlength=n_g) for _ in range(nb)]).astype(float)
            res[:, b0:b0 + nb] = 1.0 - (E @ W.T) / (R @ W.T)
        out[cell] = dict(zip(names, res))
    return out

def ci(x):
    a = (1 - CI_LEVEL) / 2
    return tuple(np.quantile(x, [a, 1 - a]))

## 4. Comprobaciones previas G1 y G2 (antes de las sondas principales)

In [ ]:
# G1: P512 bank variables alone predict the standardized chirp mass (target y, R2 against 0, global).
y_chirp = VAL["5b"]["y"][:, 0]
oof_g1 = get_oof(("y", "chirp_mass", "P512 only"), lambda: bank_features("P512"), y_chirp)
G1 = dict(r2_oof=float(r2(y_chirp, oof_g1)), min=G1_MIN)
G1["passed"] = bool(G1["r2_oof"] >= G1_MIN)
print("G1:", G1)

# G2: M14.5b residual, bank rows permuted (A variables not permuted): |Delta R2| < 0.005 on every label (global).
perm = np.random.default_rng(PERM_SEED).permutation(n)
G2 = {}
for j, lab in enumerate(LABELS):
    r = RESID["5b"][:, j]
    oa = get_oof(("5b", lab, "A"), lambda: FEAT_A["5b"], r)
    op = get_oof(("5b", lab, "A+P512perm"), lambda: np.hstack([FEAT_A["5b"], bank_features("P512")[perm]]), r)
    G2[lab] = dict(r2_A=float(r2(r, oa)), r2_Bperm=float(r2(r, op)), delta=float(r2(r, op) - r2(r, oa)))
G2_PASSED = all(abs(v["delta"]) < G2_MAX for v in G2.values())
display(pd.DataFrame(G2).T)
print("G2:", "PASA" if G2_PASSED else "FALLA")
GATES = dict(G1=G1, G2=dict(per_label=G2, max_abs_delta=G2_MAX, passed=G2_PASSED), G3=fmeta["gate_G3"])
require(G1["passed"] and G2_PASSED and fmeta["gate_G3"]["passed"],
        "Gate failed: fix and repeat without looking at the main result (pre-registration)")

## 5. Sondas principales: A y B (A + banco) para cada residuo, etiqueta y banco

In [ ]:
SETS = ["A"] + [f"A+{b}" for b in BANKS]
def set_features(m, s):
    return FEAT_A[m] if s == "A" else np.hstack([FEAT_A[m], bank_features(s[2:])])
t_all = time.perf_counter()
for m in ["5b", "5a"]:
    for j, lab in enumerate(LABELS):
        for s in SETS:
            get_oof((m, lab, s), lambda m=m, s=s: set_features(m, s), RESID[m][:, j])
print(f"total {(time.perf_counter() - t_all) / 60:.1f} min")

In [ ]:
# Paired bootstrap: every (model, label, set) series shares the row draws of its cell.
SERIES = {(m, lab, s): (RESID[m][:, j], OOF[(m, lab, s)])
          for m in ["5b", "5a"] for j, lab in enumerate(LABELS) for s in SETS}
BOOT = boot_r2(SERIES)

rows = []
for m in ["5b", "5a"]:
    for j, lab in enumerate(LABELS):
        r = RESID[m][:, j]
        for cell, mask in CELLS.items():
            bA = BOOT[cell][(m, lab, "A")]
            pA = r2(r, OOF[(m, lab, "A")], mask)
            rows.append(dict(model=m, label=lab, cell=cell, set="A", r2=pA, r2_lo=ci(bA)[0], r2_hi=ci(bA)[1],
                             d=np.nan, d_lo=np.nan, d_hi=np.nan))
            for b in BANKS:
                bB = BOOT[cell][(m, lab, f"A+{b}")]
                pB = r2(r, OOF[(m, lab, f"A+{b}")], mask)
                rows.append(dict(model=m, label=lab, cell=cell, set=f"A+{b}", r2=pB, r2_lo=ci(bB)[0], r2_hi=ci(bB)[1],
                                 d=pB - pA, d_lo=ci(bB - bA)[0], d_hi=ci(bB - bA)[1]))
TABLE = pd.DataFrame(rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.max_rows", 200):
    display(TABLE[TABLE.model == PRIMARY])

## 6. Criterio primario (M14.5b, P512, 𝓜 y χ_eff en D8–D10; ΔR² = R²_B − R²_A)

In [ ]:
prim = TABLE[(TABLE.model == CRIT["model"]) & (TABLE.set == f"A+{CRIT['bank']}") & (TABLE.cell == CRIT["cell"])
             & TABLE.label.isin(CRIT["labels"])].set_index("label")[["d", "d_lo", "d_hi"]]
if (prim.d_lo >= CRIT["relevant_lo"]).any():
    VERDICT = "margen relevante"
elif (prim.d_hi < CRIT["none_hi"]).all():
    VERDICT = "sin margen relevante"
else:
    VERDICT = "no concluyente"
with pd.option_context("display.float_format", "{:.4f}".format):
    display(prim)
print("Criterio primario:", VERDICT)

## 7. Secundarios (descriptivos): ΔR² frente a K, L32 frente a P32, M14.5a, M_tot, R²_A

In [ ]:
# L32 vs P32: R2_(A+L32) - R2_(A+P32), paired.
rows = []
for m in ["5b", "5a"]:
    for j, lab in enumerate(LABELS):
        r = RESID[m][:, j]
        for cell, mask in CELLS.items():
            dd = BOOT[cell][(m, lab, "A+L32")] - BOOT[cell][(m, lab, "A+P32")]
            rows.append(dict(model=m, label=lab, cell=cell,
                             d=r2(r, OOF[(m, lab, "A+L32")], mask) - r2(r, OOF[(m, lab, "A+P32")], mask),
                             d_lo=ci(dd)[0], d_hi=ci(dd)[1]))
L_VS_P = pd.DataFrame(rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.max_rows", 200):
    print("R²(A+L32) − R²(A+P32)")
    display(L_VS_P)
    print("R²_A por sí mismo (fracción recuperable recalibrando la salida)")
    display(TABLE[TABLE.set == "A"].pivot_table(index=["model", "label"], columns="cell", values="r2")[list(CELLS)])
    print("ΔR² por banco (todas las celdas)")
    display(TABLE[TABLE.set != "A"].pivot_table(index=["model", "label", "set"], columns="cell", values="d")[list(CELLS)])

In [ ]:
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True)
Ks = [32, 128, 512]
for i, m in enumerate(["5b", "5a"]):
    for j, lab in enumerate(LABELS):
        ax = axes[i, j]
        for c, cell in enumerate(CELLS):
            t = TABLE[(TABLE.model == m) & (TABLE.label == lab) & (TABLE.cell == cell)].set_index("set")
            dx = (c - 1.5) * 0.04
            P = t.loc[[f"A+P{k}" for k in Ks]]
            ax.errorbar(np.log2(Ks) + dx, P.d, yerr=[P.d - P.d_lo, P.d_hi - P.d], marker="o", capsize=2, label=cell,
                        color=f"C{c}")
            L = t.loc["A+L32"]
            ax.errorbar(np.log2(32) + dx - 0.3, L.d, yerr=[[L.d - L.d_lo], [L.d_hi - L.d]], marker="s", mfc="none",
                        capsize=2, color=f"C{c}")
        ax.axhline(0, color="k", lw=0.8)
        for yv, ls in ((CRIT["relevant_lo"], "--"), (CRIT["none_hi"], ":")):
            ax.axhline(yv, color="0.5", lw=0.8, ls=ls)
        ax.set(title=f"M14.{m} — {lab}", xticks=np.log2(Ks), xticklabels=[str(k) for k in Ks])
        if j == 0:
            ax.set_ylabel("ΔR²_oof = R²(A+banco) − R²(A)")
        if i == 1:
            ax.set_xlabel("K (círculos: P; cuadrado a la izquierda de 32: L32)")
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
FIG_PATH = FIG_DIR / f"M14_6_delta_r2_vs_K_{STAMP}.png"
fig.savefig(FIG_PATH, dpi=130)
plt.show()

## 8. Guardado (JSON de resumen y predicciones fuera de partición; nunca sobrescribe)

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
oof_path = OUT_DIR / f"M14_6_oof_{STAMP}.npz"
keys = list(OOF)
FOLD_ID = np.empty(n, np.int8)
for i, (_, te) in enumerate(FOLDS):
    FOLD_ID[te] = i
with oof_path.open("xb") as f:
    np.savez(f, row_ids=row_ids, oof=np.stack([OOF[k] for k in keys]), keys=np.array(["|".join(k) for k in keys]),
             folds=FOLD_ID,
             perm_G2=perm, tau_decile=DEC)
summary = dict(
    experiment="M14.6", created_utc=STAMP, preregistration="docs/experiments/M14_6_bank_probes.md",
    verdict=VERDICT, primary=prim.reset_index().to_dict("records"), criterion=CRIT, gates=GATES,
    inputs=dict(features=str(FEATURES_PATH), features_sha256=file_sha256(FEATURES_PATH),
                features_metadata=fmeta, val={m: dict(path=d["path"], sha256=d["sha256"]) for m, d in VAL.items()},
                cal_5b=dict(path=CAL5B["path"], sha256=CAL5B["sha256"]), label_stats=str(LABEL_STATS_PATH)),
    settings=dict(hgb=HGB_PARAMS, n_folds=N_FOLDS, fold_seed=FOLD_SEED, n_boot=N_BOOT, ci_level=CI_LEVEL,
                  boot_seed=BOOT_SEED, perm_seed=PERM_SEED, a_variables="mu(3), sigma(3), corr(Sigma)(3)",
                  bank_variables="per template: peak H1, L1, V1, network", sklearn=sklearn.__version__),
    groups=dict(tau_pred_edges_cal_5b_s=TAU_EDGES_CAL.tolist(), tau_groups=TAU_GROUPS,
                counts={g: int(m.sum()) for g, m in CELLS.items()}),
    table=TABLE.to_dict("records"), l32_vs_p32=L_VS_P.to_dict("records"),
    fit_minutes={"|".join(k): v / 60 for k, v in FIT_TIME.items()},
    outputs=dict(oof=str(oof_path), figure=str(FIG_PATH)), checks=CHECKS)
sum_path = OUT_DIR / f"M14_6_summary_{STAMP}.json"
with sum_path.open("x") as f:
    json.dump(summary, f, indent=2, default=float)
print("Saved:", sum_path, "|", oof_path)

## Lectura